In [1]:
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from os import environ

# Splitting documents

In [19]:
# Load pdf documents
documents_1 = ''

reader = PdfReader('../data/product_description.pdf')
for page in reader.pages:
    documents_1 += page.extract_text()
print(f"page numbers: {len(reader.pages)}")

page numbers: 6


In [ ]:
print(documents_1[:500])

LangFund  Secured Loan Product Description 
 
Dreaming of a home renovation, needing to boost your business cash flow, or looking to 
consolidate debt? The LangFund Secured Loan is designed to help you achieve your 
financial goals with confidence and flexibility. 
 
Key Features & Highlights: 
Loan


In [ ]:
documents_1[:500]

'LangFund  Secured Loan Product Description \n \nDreaming of a home renovation, needing to boost your business cash flow, or looking to \nconsolidate debt? The LangFund Secured Loan is designed to help you achieve your \nfinancial goals with confidence and flexibility. \n \nKey Features & Highlights: \nLoan Amount Range: Borrow from $1,000 to $30,000 to cover your essential needs or \naspirational projects. \nCompetitive Interest Rates: Benefit from an attractive Annual Percentage Rate (APR) \nranging from'

In [20]:
# Remove the \n and retain \n \n
documents_1 = documents_1.replace("\n \n", "new_paragraph")
documents_1 = documents_1.replace("\n", " ")
documents_1 = documents_1.replace("new_paragraph", "\n \n")

In [21]:
# Document Splitting
chunk_size = 250
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n \n", " ", ""]
)
split_1 = splitter.split_text(documents_1)
split_1 = splitter.create_documents(split_1)

In [22]:
split_1[:5]

[Document(metadata={}, page_content='LangFund  Secured Loan Product Description'),
 Document(metadata={}, page_content='Dreaming of a home renovation, needing to boost your business cash flow, or looking to  consolidate debt? The LangFund Secured Loan is designed to help you achieve your  financial goals with confidence and flexibility.'),
 Document(metadata={}, page_content='Key Features & Highlights:  Loan Amount Range: Borrow from $1,000 to $30,000 to cover your essential needs or  aspirational projects.  Competitive Interest Rates: Benefit from an attractive Annual Percentage Rate (APR)  ranging from 5% to 12%,'),
 Document(metadata={}, page_content='Percentage Rate (APR)  ranging from 5% to 12%, ensuring affordable repayments.  Fixed Loan Term: Enjoy a clear and manageable 2-year repayment period, allowing you to  plan your finances effectively.  Flexible Repayment Frequency: Choose the option'),
 Document(metadata={}, page_content='Flexible Repayment Frequency: Choose the option 

# Embedding and storing

In [ ]:
# # Download embeddings model
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\USER\miniconda3\envs\venv_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# # Download and save the embeddings model
# from sentence_transformers import SentenceTransformer

# # Load model from Hugging Face (downloads if not cached)
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # Save model locally
# model.save("../saved_models/all-MiniLM-L6-v2")

In [ ]:
# Load the saved embeddings model
embeddings = HuggingFaceEmbeddings(model_name="../saved_models/all-MiniLM-L6-v2")

In [ ]:
# Implement embeddings
db = FAISS.from_documents(split_1, embeddings)

# Save db
db.save_local('../vector_store/product_description')

# Similarity search

In [ ]:
# Load db
load_db = FAISS.load_local(
    '../vector_store/product_description/', embeddings, allow_dangerous_deserialization=True
)

In [ ]:
# Retrieve answer
question = 'can I buy an apartment using this loan?'

search = load_db.similarity_search(question)
search

[Document(id='82f10ee7-b911-4ce7-b0dd-e18888ac3002', metadata={}, page_content='the new loan payments are manageable within your current financial  obligations (typically aiming for a DTI below 30%).  Home Ownership: Not always a strict requirement, but homeownership can strengthen  your application, especially for larger loan'),
 Document(id='dd7a5b5e-0988-4c14-9f60-035d9f49ade6', metadata={}, page_content='approval! You’ll receive clear  communication regarding your loan offer, including the approved amount, interest rate,  repayment schedule, and all associated terms and conditions. Once you accept the offer,  the funds are swiftly disbursed to your'),
 Document(id='997a6546-3e9b-47d9-aa0c-49be98cb32cd', metadata={}, page_content='Key Features & Highlights:  Loan Amount Range: Borrow from $1,000 to $30,000 to cover your essential needs or  aspirational projects.  Competitive Interest Rates: Benefit from an attractive Annual Percentage Rate (APR)  ranging from 5% to 12%,'),
 Document

In [ ]:
# Query more or less text chunks
search = load_db.similarity_search(question, k=6)
search

[Document(id='82f10ee7-b911-4ce7-b0dd-e18888ac3002', metadata={}, page_content='the new loan payments are manageable within your current financial  obligations (typically aiming for a DTI below 30%).  Home Ownership: Not always a strict requirement, but homeownership can strengthen  your application, especially for larger loan'),
 Document(id='dd7a5b5e-0988-4c14-9f60-035d9f49ade6', metadata={}, page_content='approval! You’ll receive clear  communication regarding your loan offer, including the approved amount, interest rate,  repayment schedule, and all associated terms and conditions. Once you accept the offer,  the funds are swiftly disbursed to your'),
 Document(id='997a6546-3e9b-47d9-aa0c-49be98cb32cd', metadata={}, page_content='Key Features & Highlights:  Loan Amount Range: Borrow from $1,000 to $30,000 to cover your essential needs or  aspirational projects.  Competitive Interest Rates: Benefit from an attractive Annual Percentage Rate (APR)  ranging from 5% to 12%,'),
 Document

In [ ]:
search_scores = load_db.similarity_search_with_score(question)
search_scores

[(Document(id='82f10ee7-b911-4ce7-b0dd-e18888ac3002', metadata={}, page_content='the new loan payments are manageable within your current financial  obligations (typically aiming for a DTI below 30%).  Home Ownership: Not always a strict requirement, but homeownership can strengthen  your application, especially for larger loan'),
  1.0274754),
 (Document(id='dd7a5b5e-0988-4c14-9f60-035d9f49ade6', metadata={}, page_content='approval! You’ll receive clear  communication regarding your loan offer, including the approved amount, interest rate,  repayment schedule, and all associated terms and conditions. Once you accept the offer,  the funds are swiftly disbursed to your'),
  1.1242442),
 (Document(id='997a6546-3e9b-47d9-aa0c-49be98cb32cd', metadata={}, page_content='Key Features & Highlights:  Loan Amount Range: Borrow from $1,000 to $30,000 to cover your essential needs or  aspirational projects.  Competitive Interest Rates: Benefit from an attractive Annual Percentage Rate (APR)  rangi

In [ ]:
load_db.similarity_search_with_score(question, k=8)

[(Document(id='82f10ee7-b911-4ce7-b0dd-e18888ac3002', metadata={}, page_content='the new loan payments are manageable within your current financial  obligations (typically aiming for a DTI below 30%).  Home Ownership: Not always a strict requirement, but homeownership can strengthen  your application, especially for larger loan'),
  1.0274754),
 (Document(id='dd7a5b5e-0988-4c14-9f60-035d9f49ade6', metadata={}, page_content='approval! You’ll receive clear  communication regarding your loan offer, including the approved amount, interest rate,  repayment schedule, and all associated terms and conditions. Once you accept the offer,  the funds are swiftly disbursed to your'),
  1.1242442),
 (Document(id='997a6546-3e9b-47d9-aa0c-49be98cb32cd', metadata={}, page_content='Key Features & Highlights:  Loan Amount Range: Borrow from $1,000 to $30,000 to cover your essential needs or  aspirational projects.  Competitive Interest Rates: Benefit from an attractive Annual Percentage Rate (APR)  rangi

# Answer retrieval and generation

In [5]:
# Load environment
load_dotenv()
api_key_groq = environ['API_GROG']

In [6]:
# Load LLM
llm = ChatGroq(
    model="llama3-70b-8192",
    temperature=0,
    max_tokens=40,
    api_key=api_key_groq
)

In [7]:
# Create the chatbot
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=load_db.as_retriever(),
    return_source_documents=True,
)

In [8]:
# Ask a question
question = 'what is the loan name?'
response = qa({'query': question})
response

C:\Users\USER\AppData\Local\Temp\ipykernel_32288\815458679.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa({'query': question})


{'query': 'what is the loan name?',
 'result': 'The loan name is LangFund Secured Loan.',
 'source_documents': [Document(id='f9c73f68-35eb-41b3-882c-c78cc64b7cf2', metadata={}, page_content='minimal and  transparently outlined in your loan agreement. We empower you to take control of your  finances and reduce your interest burden when it suits you.  We are committed to providing a loan experience that is not only convenient but also'),
  Document(id='dd7a5b5e-0988-4c14-9f60-035d9f49ade6', metadata={}, page_content='approval! You’ll receive clear  communication regarding your loan offer, including the approved amount, interest rate,  repayment schedule, and all associated terms and conditions. Once you accept the offer,  the funds are swiftly disbursed to your'),
  Document(id='b91db590-43ca-495b-b91f-8371f8852477', metadata={}, page_content='loan amount aligns with your repayment  capacity, safeguarding both your financial well-being and ours.'),
  Document(id='6a0d8c9e-0927-4de2-8ac9-

In [9]:
# Ask a question
question = 'Can the loan fund school fees?'
response = qa({'query': question})
response

{'query': 'Can the loan fund school fees?',
 'result': 'Yes, the Educational Loan can be used to cover tuition fees, among other educational expenses.',
 'source_documents': [Document(id='a57a50bd-fc8f-4f84-83aa-14c0f5e46c2a', metadata={}, page_content='Educational Loan: Invest in your future! Cover tuition fees, living expenses, and study  materials for undergraduate, postgraduate, and vocational courses.  Medical Loan: Access timely healthcare. Finance planned surgeries, emergency  treatments,'),
  Document(id='19017054-779a-4d9a-a891-fa45456c07d3', metadata={}, page_content="to achieve your goals. We understand that every individual's  journey is unique, which is why we offer tailored solutions with transparent terms and a  commitment to your financial well-being.  Loan Intents:  Educational Loan: Invest in your future!"),
  Document(id='997a6546-3e9b-47d9-aa0c-49be98cb32cd', metadata={}, page_content='Key Features & Highlights:  Loan Amount Range: Borrow from $1,000 to $30,000 to c

## LCEL

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [15]:
system_prompt = """
Answer based on the context: {context}.
If the the context does not provide the answer to the query, admit it. The answer is less than 50 words.
"""

chat_promp_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

retriever = load_db.as_retriever(search_kwargs={"k": 5})

def join_text(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chain = (
    {"context": retriever | join_text, "question": RunnablePassthrough()}
    | chat_promp_template
    | llm
    | StrOutputParser()
)

In [28]:
response = rag_chain.invoke("Can the loan fund school fees?")
response

'Yes, the Educational Loan can fund school fees, as it covers tuition fees, living expenses, and study materials for undergraduate, postgraduate, and vocational courses.'

In [17]:
response = rag_chain.invoke("how much is the LangFund profit this year?")
response

"The context does not provide the answer to this query. The product description only highlights the features and benefits of the LangFund Secured Loan, but does not mention the company's profit."